# Extract mantle data

This notebook will extract mantle diagnostics from plate-model-driven G-ADOPT outputs, using the training points extracted by notebooks `00b` or `00bb`. The resultant data is saved or appended to `training_data_global_mantle.csv`. This can then be used to train the models in later notebooks (`01*.ipynb`).

G-ADOPT outputs have been interpolated into (lat, lon, depth, time) grids from original volumetric .xy files.

## Notebook setup

These cells set paths and run parameters from the selected config file.

### Config

In [1]:
config_file = "config/.run_config.yml"

In [2]:
from pathlib import Path

from lib.paths import PathConfigManager

pcm = PathConfigManager(config_file, notebook="00d")

# =====================
# Filestructure
# =====================

plate_model_dir = pcm.PLATE_MODEL_DIR
extracted_data_dir = pcm.EXTRACTED_DATA_DIR
points_output_dir = pcm.POINTS_DATA_DIR

pcm.create_directories()

# =====================
# Notebook scope
# =====================

use_features = pcm.use_features

# =====================
# Plate model
# =====================

plate_model_name = pcm.config["plate_model"]["plate_model_name"]
use_provided_plate_model = pcm.use_provided_plate_model

# =====================
# Run parameters
# =====================

use_extracted_data = pcm.use_extracted_data
n_jobs = pcm.config["n_jobs"]
overwrite = pcm.config["overwrite_output"]
verbose = pcm.config["verbose"]
min_time = pcm.config["timespan"]["min"]
max_time = pcm.config["timespan"]["max"]
times = range(min_time, max_time + 1)
random_seed = pcm.config["random_seed"]
use_mantle_features = pcm.config.get("use_mantle_features", False)

## Notebook setup

Imports, definitions, etc.

### Imports

In [3]:
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    from gplately.tools import plate_isotherm_depth

from lib.check_files import (
    check_prepared_data,
    check_plate_model,
)
from lib.plate_models import get_plate_reconstruction

from lib.sample_mantle import extract_basic_mantle_features

# Suppress occasional joblib warnings
%env PYTHONWARNINGS=ignore::UserWarning
warnings.simplefilter("ignore", UserWarning)

env: PYTHONWARNINGS=ignore::UserWarning


### Input and output files

If necessary, the plate model will be downloaded:

In [4]:
if use_provided_plate_model:
    check_plate_model(plate_model_dir, verbose=True)
    plate_model_name = None
plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

2026-03-24,00:11:09 - pmm - WARNING - Unable to fetch https://repo.gplates.org/webdav/pmm/config/models_v2.json.
2026-03-24,00:11:09 - pmm - WARNING - Unable to fetch https://www.earthbyte.org/webdav/pmm/config/models_v2_eb.json.
2026-03-24,00:11:09 - pmm - WARNING - Unable to fetch https://portal.gplates.org/static/pmm/config/models_v2_gp.json.


In [5]:
if use_extracted_data:
    data_dir = extracted_data_dir
    if verbose:
        print(f"Using extracted training data under {data_dir}")
else:
    data_dir = check_prepared_data("prepared_data", verbose=True)
    if not isinstance(data_dir, Path):
        data_dir = Path(data_dir)

mantle_data_dir = pcm.MANTLE_DATA_DIR

_candidate = extracted_data_dir / "training_data_no_mantle.csv"
training_data_filename = _candidate if _candidate.exists() else pcm.TRAINING_DATA_PATH
training_output_filename = points_output_dir / "training_data_with_mantle.csv"

_candidate = extracted_data_dir / "grid_data_no_mantle.csv"
grid_data_filename = _candidate if _candidate.exists() else pcm.GRID_DATA_PATH
grid_output_filename = points_output_dir / "grid_data_with_mantle.csv"



Using extracted training data in /scratch/xd2/me5758/PUB-framework-Alfonso/data/zahirovic2022/extracted_data/trenches_6.0_deg_buffer


## Training data extraction

In [6]:
if use_features('mantle'):
    # Load training data
    training_data = pd.read_csv(training_data_filename)

### Extract simple mantle features

Reconstruct labelled points to sample mantle model outputs at various depths. Resultant data are appended to `training_data_global.csv`. When sampling, linear interpolation is used to minimise distortion from temporal sparseness of the mantle grids.

In [9]:
if use_features('mantle'):
    depths_km = npcm.arange(100, 2900, 100) # km
    mantle_fields = (
        # "FullTemperature_CG",
        "Pressure",
        "Radial_Velocity",
        # "Temperature_CG",
        "Temperature_Deviation_CG",
        "Velocity_x",
        "Velocity_y",
        "Velocity_z",
        "Viscosity_CG",
    )

    training_data = extract_basic_mantle_features(
        points=training_data,
        mantle_dir=mantle_data_dir,
        depths_km=depths_km,
        features_to_extract=mantle_fields,
    )

### Extract tangential velocities

This cell will reconstruct labelled points to calculate cumulative mantle diagnostics at various depths of the mantle model outputs. Resultant data are appended to `training_data_global.csv`. Linear interpolation is used to sample these outputs to minimise distortion from temporal sparseness of the mantle grids.

In [ ]:
if use_features('mantle'):
    def extract_tangential_mantle_velocities():
        ...

### Extract cumulative mantle features

This cell will reconstruct labelled points to calculate cumulative mantle diagnostics at various depths of the mantle model outputs. Resultant data are appended to `training_data_global.csv`. Linear interpolation is used to sample these outputs to minimise distortion from temporal sparseness of the mantle grids.

In [10]:
if use_features('mantle'):
    def extract_cumulative_mantle_features():
        ...

### Extract other shenanigans

And more, and more, and more!! What joy we have, living like leeches mawed to the capillaries of progress.

In [11]:
if use_features('mantle'):
    def extract_some_other_nonsense_too():
        ...

### Save to file

Finally, we write the dataset to a CSV file.

If `use_mantle_features == True`, rename the training data with mantle features to `training_data_global.csv`, enabling later notebooks (`01*` onwards) to easily read training data.

In [12]:
if use_features('mantle'):
    training_data.to_csv(training_output_filename, index=False)

    if use_mantle_features:
        # Backup old data
        try:
            pcm.TRAINING_DATA_PATH.rename(
                pcm.TRAINING_DATA_PATH.with_name("training_data_no_mantle.csv")
            )
        except FileNotFoundError:
            pass
        # Save new data with mantle features
        training_data.to_csv(pcm.TRAINING_DATA_PATH, index=False)

    training_data.groupby(["region", "label"]).size()

region          label     
East Asia       negative        14
                positive       159
                unlabelled    7729
North America   negative        61
                positive       293
                unlabelled    8104
Other           negative       214
                positive         2
                unlabelled    6476
South America   negative      1389
                positive       275
                unlabelled    7845
Southeast Asia  negative         4
                positive       145
                unlabelled    5457
Tethys          negative        21
                positive       481
                unlabelled    6507
dtype: int64

## Grid data extraction

### Extract simple mantle features

Reconstruct labelled points to sample mantle model outputs at various depths. Resultant data are appended to `training_data_global.csv`. When sampling, linear interpolation is used to minimise distortion from temporal sparseness of the mantle grids.

In [ ]:
if use_features('mantle'):
    depths_km = npcm.arange(100, 2900, 100) # km
    mantle_fields = (
        # "FullTemperature_CG",
        "Pressure",
        "Radial_Velocity",
        # "Temperature_CG",
        "Temperature_Deviation_CG",
        "Velocity_x",
        "Velocity_y",
        "Velocity_z",
        "Viscosity_CG",
    )

    grid_data = pd.read_csv(grid_data_filename)
    grid_data = extract_basic_mantle_features(
        points=grid_data,
        mantle_dir=mantle_data_dir,
        depths_km=depths_km,
        features_to_extract=mantle_fields,
    )

### Extract cumulative mantle features

This cell will reconstruct labelled points to calculate cumulative mantle diagnostics at various depths of the mantle model outputs. Resultant data are appended to `training_data_global.csv`. Linear interpolation is used to sample these outputs to minimise distortion from temporal sparseness of the mantle grids.

In [ ]:
if use_features('mantle'):
    def extract_cumulative_mantle_features():
        ...

### Extract other shenanigans

And more, and more, and more!! What joy we have, living like leeches mawed to the capillaries of progress.

In [ ]:
if use_features('mantle'):
    def extract_some_other_nonsense_too():
        ...

### Save to file

Finally, we write the dataset to a CSV file.

If `use_mantle_features == True`, substitute the grid data with mantle features for `grid_data.csv`, letting later notebooks read it (`01*` onwards).

In [ ]:
if use_features('mantle'):
    grid_data.to_csv(grid_output_filename, index=False)

    if use_mantle_features:
        # Backup old data
        try:
            pcm.GRID_DATA_PATH.rename(
                pcm.GRID_DATA_PATH.with_name("grid_data_no_mantle.csv")
            )
        except FileNotFoundError:
            pass
        # Save new data with mantle features
        grid_data.to_csv(pcm.GRID_DATA_PATH, index=False)

    grid_data.groupby(["region", "label"]).size()

region          label     
East Asia       negative        14
                positive       159
                unlabelled    7729
North America   negative        61
                positive       293
                unlabelled    8104
Other           negative       214
                positive         2
                unlabelled    6476
South America   negative      1389
                positive       275
                unlabelled    7845
Southeast Asia  negative         4
                positive       145
                unlabelled    5457
Tethys          negative        21
                positive       481
                unlabelled    6507
dtype: int64